# ADC Example 3 — Discontinuous Conversion Mode

**Target:** STM32F103C8T6 · ADC1 · 4 channels
**Focus:** Split a long scan sequence into smaller **sub-groups**, converted one sub-group per trigger.

---
## Concept

Normally, a scan sequence converts **all** ranks on one trigger. In discontinuous mode, the sequence is subdivided: each trigger converts only `NbrOfDiscConversion` ranks, and the next trigger continues from the next rank[reference:3].

This is useful when you want to interleave ADC sampling with other tasks, or when you want each channel's sample instant to be separated by a known trigger interval.

---
## CubeMX Configuration

### ADC1 Parameters
| Parameter | Setting |
| :--- | :--- |
| `Scan Conversion Mode` | **Enabled** (mandatory for discontinuous) |
| `Continuous Conversion Mode` | **Disabled** (mandatory — discontinuous and continuous are mutually exclusive)[reference:4] |
| `Discontinuous Conversion Mode` | **Enabled** |
| `Number Of Discontinuous Conversions` | **1** (one channel per trigger) |
| `Number Of Conversion` (total) | **4** |
| Rank 1 | Channel 0 |
| Rank 2 | Channel 1 |
| Rank 3 | Channel 4 |
| Rank 4 | Channel 5 |

### Trigger source
Select **Software Start** for manual triggering, or a timer TRGO for timed triggering. With `NbrOfDiscConversion = 1`, each trigger converts exactly **one** channel.

> ⚠️ **Discontinuous mode requires Scan Mode = Enabled.** It has no effect if scan mode is disabled[reference:5].

## Generated Initialization (key excerpt)

```c
hadc1.Init.ScanConvMode          = ADC_SCAN_ENABLE;
hadc1.Init.ContinuousConvMode    = DISABLE;
hadc1.Init.DiscontinuousConvMode = ENABLE;
hadc1.Init.NbrOfDiscConversion   = 1;
hadc1.Init.NbrOfConversion       = 4;
```

The HAL writes `ADC_CR1_DISCEN` and `ADC_CR1_DISCONTINUOUS_NUM(NbrOfDiscConversion)` into `ADC1->CR1`[reference:6].

## User Code — Software-Triggered Sub-Groups

With `NbrOfDiscConversion = 1`, each `HAL_ADC_Start()` converts one rank. The sequence position auto-advances on each trigger.

```c
uint16_t results[4];

for (int i = 0; i < 4; i++) {
    HAL_ADC_Start(&hadc1);
    HAL_ADC_PollForConversion(&hadc1, 10);
    results[i] = HAL_ADC_GetValue(&hadc1);
    // Rank 1 → Rank 2 → Rank 3 → Rank 4
}
// After 4 triggers, the sequence wraps to Rank 1
```


## User Code — Timer-Triggered Sub-Groups (Interrupt)

For precise spacing between sub-groups, use a timer TRGO. Each trigger converts one rank and fires the callback.

```c
volatile uint16_t disc_buf[4];
volatile uint8_t  disc_idx = 0;

HAL_ADC_Start_IT(&hadc1);   // enable EOC interrupt
HAL_TIM_Base_Start(&htim3); // timer TRGO drives one rank per event

void HAL_ADC_ConvCpltCallback(ADC_HandleTypeDef *hadc) {
    if (hadc->Instance == ADC1) {
        if (disc_idx < 4) {
            disc_buf[disc_idx++] = HAL_ADC_GetValue(hadc);
        }
    }
}
```


## Behaviour Summary

| | Normal Scan | Discontinuous |
| :--- | :--- | :--- |
| Conversions per trigger | All ranks | `NbrOfDiscConversion` ranks |
| Total triggers per full sequence | 1 | `NbrOfConversion / NbrOfDiscConversion` |
| `DR` overwrite risk | High without DMA | Lower (fewer results per trigger) |
| Typical use | Fast multi-channel sweep | Staggered sampling, low-power duty-cycled acquisition |